In [2]:
import time
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
import great_expectations as ge

In [7]:

from matplotlib.style import context


def run_self_healing_pipeline(data_path):
    print(f"\n==================================================")
    print(f"RUNNING INTEGRATED SELF-HEALING PIPELINE ON: {data_path}")
    print(f"==================================================")
    
    # ----------------------------------------------------
    # STAGE 0: DATA INGESTION & FILTERING
    # ----------------------------------------------------
    t_start_total = time.time()
    df_raw = pd.read_csv(data_path)
    df_filtered = df_raw[df_raw['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()
    
    # ----------------------------------------------------
    # STAGE 1: DECLARATIVE VALIDATION GATEWAY (Great Expectations)
    # ----------------------------------------------------
    t_start_valid = time.time()
    context = ge.get_context(mode="ephemeral")

    datasource = context.data_sources.add_pandas(
    name="in_memory_datasource"
)
    #datasource = context.sources.add_pandas("in_memory_datasource")
    data_asset = datasource.add_dataframe_asset(name="paysim_asset")
    batch_request = data_asset.build_batch_request(dataframe=df_filtered)
    
    suite = context.add_or_update_expectation_suite(expectation_suite_name="paysim_suite")
    validator = context.get_validator(batch_request=batch_request, expectation_suite=suite)
    
    target_cols = ['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']
    for col in target_cols:
        validator.expect_column_values_to_not_be_null(column=col)
    validator.expect_column_values_to_be_between(column='amount', min_value=0)
    
    validation_results = validator.validate()
    t_end_valid = time.time()
    
    t_valid_duration = t_end_valid - t_start_valid
    validation_passed = validation_results["success"]
    
    print(f"[Stage 1] Validation Status: {'PASSED' if validation_passed else 'FAILED (Corruptions Detected)'}")
    print(f"[Stage 1] Validation Latency (T_valid): {t_valid_duration:.4f} seconds")
    
    # ----------------------------------------------------
    # STAGE 2: PROBABILISTIC SELF-HEALING LAYER (HoloClean Logic)
    # ----------------------------------------------------
    t_start_repair = time.time()
    
    if not validation_passed:
        print("[Stage 2] Triggering Self-Healing Layer...")
        df_repaired = df_filtered.copy()
        
        # Relational Denial Constraints Execution
        mask_old = df_repaired['oldbalanceOrg'].isna() & df_repaired['newbalanceOrig'].notna() & df_repaired['amount'].notna()
        df_repaired.loc[mask_old, 'oldbalanceOrg'] = df_repaired.loc[mask_old, 'newbalanceOrig'] + df_repaired.loc[mask_old, 'amount']
        
        mask_new = df_repaired['newbalanceOrig'].isna() & df_repaired['oldbalanceOrg'].notna() & df_repaired['amount'].notna()
        df_repaired.loc[mask_new, 'newbalanceOrig'] = df_repaired.loc[mask_new, 'oldbalanceOrg'] - df_repaired.loc[mask_new, 'amount']
        
        mask_amt = df_repaired['amount'].isna() & df_repaired['oldbalanceOrg'].notna() & df_repaired['newbalanceOrig'].notna()
        df_repaired.loc[mask_amt, 'amount'] = df_repaired.loc[mask_amt, 'oldbalanceOrg'] - df_repaired.loc[mask_amt, 'newbalanceOrig']
        
        # Fallback Imputations
        for col in ['oldbalanceDest', 'newbalanceDest']:
            df_repaired[col] = df_repaired[col].fillna(df_repaired[col].median())
        for col in ['amount', 'oldbalanceOrg', 'newbalanceOrig']:
            df_repaired[col] = df_repaired[col].fillna(df_repaired[col].median())
    else:
        print("[Stage 2] Data passed validation assertions. Skipping repair layer.")
        df_repaired = df_filtered.copy()
        
    t_end_repair = time.time()
    
    # Isolate repair cost equation: Delta T_repair = T_sanitized - T_valid
    delta_t_repair = t_end_repair - t_start_repair
    print(f"[Stage 2] Repair Overhead (Delta T_repair): {delta_t_repair:.4f} seconds")
    
    # ----------------------------------------------------
    # STAGE 3: DOWNSTREAM ANOMALY MODEL (Isolation Forest)
    # ----------------------------------------------------
    df_repaired['balance_diff_orig'] = (
        df_repaired['oldbalanceOrg'] - df_repaired['amount'] - df_repaired['newbalanceOrig']
    )
    
    feature_cols = ['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'balance_diff_orig']
    X = df_repaired[feature_cols]
    y = df_repaired['isFraud']
    
    model = IsolationForest(
        n_estimators=100,
        contamination=y.mean(),
        random_state=42,
        n_jobs=-1
    )
    
    preds = model.fit_predict(X)
    preds_binary = (preds == -1).astype(int)
    
    t_end_total = time.time()
    delta_t_total = t_end_total - t_start_total
    
    # Metrics
    precision = precision_score(y, preds_binary, pos_label=1, zero_division=0)
    recall = recall_score(y, preds_binary, pos_label=1, zero_division=0)
    f1 = f1_score(y, preds_binary, pos_label=1, zero_division=0)
    
    print(f"\n=== FINAL PIPELINE METRICS ===")
    print(f"Total Pipeline Latency (Delta T_total): {delta_t_total:.4f} seconds")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")
    
    return {
        "dataset": data_path,
        "delta_t_repair": delta_t_repair,
        "delta_t_total": delta_t_total,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

# Execute comparison across datasets
if __name__ == "__main__":
    datasets = ['Dataset.csv', 'paysim_noise_5.csv', 'paysim_noise_10.csv', 'paysim_noise_20.csv']
    results = []
    
    for path in datasets:
        try:
            res = run_self_healing_pipeline(path)
            results.append(res)
        except FileNotFoundError:
            print(f"File {path} not found. Skipping...")
            
    summary_df = pd.DataFrame(results)
    print("\n================ FINAL COMPARATIVE SUMMARY ================")
    print(summary_df.to_string(index=False))


RUNNING INTEGRATED SELF-HEALING PIPELINE ON: Dataset.csv


TypeError: DataFrameAsset.build_batch_request() got an unexpected keyword argument 'dataframe'